## Prompt Engineering

一般来说，系统提示词 (System Prompt) 会包含以下几个部分，通常按此顺序排列:
- 身份角色 (ldentity): 描述AI的职责、沟通风格和总体目标.
- 指令说明 (instructions): 请指导模型如何生成所需的响应. 它应该遵循哪些规则? 模型应该做什么, 以及模型绝对不能做什么?
- 对话示例 (Examples): 提供可能的输入示例，以及型期望的输出.
- 背景信息 (Context): 向模型提供生成响应所需的任何额外信息, 例如RAG的额外知识库数据, 或您认为特别相关的任何其他数据.

在编写 System Prompt 时，可以使用 Markdown 格式和 XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界:
- Markdown 的标题和列表有助于标记提示的不同部分, 并向模型传达层级结构. 它们还可以提高开发过程中提示的可读性.
- XML 标签可以帮助明确区分一段内容 (例如用作参考的辅助文档) 的起始和结束位置.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

system_prompt = """
	# 身份
	- 你是一个科幻作家, 根据用户的要求创建一个太空之都
    
	# 指令
	- 请务必以JSON格式输出, 不要加任何 markdown 样式
    
	# 示例
	user: 月球的首都是什么?
	assistant: {
  		"name": "月华市 (Lunaria)",
    	"location": "位于月球正面赤道附近的静海基地遗址之上, 依托巨大的穹顶与地下网络建成",
    	"vibe": "冷冽, 高效, 革新",
    	"economy": "氦-3能源开采, 量子通信枢纽, 尖端生物圈农业"
    }
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Who are you?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

## 结构化输出

In [1]:
from pydantic import BaseModel

class capitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model="gpt-4o-mini",
    system_prompt="You are a sci-fi writer. Please build a space capital city based on the user's request.",
    response_format=capitalInfo
)

response = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of the Moon?")]},
)

print(response)

{'messages': [HumanMessage(content='What is the capital of the Moon?', additional_kwargs={}, response_metadata={}, id='9a0efa8e-c39c-423c-a837-eb1e1f2306a8'), AIMessage(content='{"name":"Lunaris","location":"The Sea of Tranquility, Moon","vibe":"Futuristic and Serene","economy":"Lunar Tourism and Resource Extraction"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 114, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f51aa69871', 'id': 'chatcmpl-EQOItQK8B8jwoGVLjRDdcBPv5kSqQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}